### Теоретическое обоснование: Почему Градиентный Бустинг, а не другие семейства моделей?

При проектировании ML-архитектуры мы сознательно отказались от линейных моделей и тяжелых нейронных сетей в пользу деревьев решений (LightGBM, CatBoost, XGBoost). На это есть 4 фундаментальные математические и инженерные причины:

**1. Топология пространственных данных (Почему не линейные модели?)**
Географические координаты ($coord\_x$, $coord\_y$) и геометрические метрики (площадь, периметр) имеют абсолютно нелинейную связь с целевой переменной. Линейная регрессия пытается найти глобальные веса для признаков:
$$\hat{y} = w_1 \cdot X + w_2 \cdot Y + b$$
На карте это не имеет физического смысла. Деревья решений делают ортогональные разбиения пространства признаков:
$$X > x_{threshold} \land Y < y_{threshold}$$
На плоскости геоданных это буквально означает выделение прямоугольных географических кластеров и микрорайонов застройки. Деревья «понимают» карту из коробки.

**2. Превосходство на гетерогенных табличных данных (Почему не Нейросети?)**
В задачах с табличными данными (Tabular Data), где признаки имеют разную физическую природу (координаты, метры, штуки этажей, категориальные теги), градиентный бустинг математически превосходит полносвязные нейронные сети. Нейросети требуют идеального масштабирования признаков (например, Z-score нормализации):
$$x'_{i} = \frac{x_i - \mu}{\sigma}$$
В противном случае градиенты затухают или взрываются. Деревья решений инвариантны к монотонным преобразованиям и не требуют нормализации признаков, что сохраняет исходное физическое распределение данных.

**3. Нативная работа с категориальными признаками и пропусками**
Наш датасет содержит высококардинальные категории (`ml_category`) и пустоты ($NaN$ в этажности пристроек). 
* Использование классического One-Hot Encoding раздуло бы размерность матрицы до пространства $\mathbb{R}^N$, приведя к «проклятию размерности». Алгоритмы вроде CatBoost используют *Target Encoding* под капотом.
* Пропуски не требуют заполнения средним или медианой. Бустинг сам определяет оптимальное направление (в левое или правое поддерево) для значений $NaN$ на основе минимизации градиента функции потерь.

**4. Робастность к выбросам через функцию потерь**
В кадастровых данных всегда есть аномалии (небоскребы среди пятиэтажек, ошибки разметки). Мы оптимизируем модель под функцию потерь **MAE** ($L_1$-норма), а не классическую MSE. 
$$MAE = \frac{1}{N} \sum_{i=1}^{N} |y_i - \hat{y}_i|$$
Производная (вектор антиградиента) для $L_1$ loss равна константе:
$$\frac{\partial L}{\partial \hat{y}_i} = \mathrm{sign}(\hat{y}_i - y_i)$$
Это означает, что экстремальные выбросы (вроде «Лахта Центра») не создают гигантских градиентов и не «ломают» веса модели, позволяя модели стабильно сходиться к медианным значениям высоты для типичной застройки.

### Выбор и обоснование ML-модели

**Кандидаты:**
В качестве базовых алгоритмов мы рассматривали 3 фреймворка градиентного бустинга: `CatBoost`, `XGBoost` и `LightGBM`. 

**Методология:**
* **Оптимизация:** Для каждой модели был проведен независимый отбор признаков, а гиперпараметры оптимизировались методами байесовского поиска через фреймворк **Optuna**.
* **Оценка:** Сравнение качества производилось на кросс-валидации по трем метрикам: **MAE** (целевая бизнес-метрика), **RMSE** (контроль выбросов) и **R²** (уверенность модели).

**Результаты и отказ от ансамблирования:**
По результатам честной валидации наилучшее качество показала модель **LightGBM**. 

В ходе экспериментов мы также рассматривали стратегию объединения предсказаний (Stacking/Blending). Построенная матрица корреляций выявила коэффициент Пирсона **> 0.99** между ответами всех трех алгоритмов. Это доказывает, что модели извлекли идентичные паттерны из геоданных. 

**Итоговый вывод:**
При такой высокой корреляции ансамблирование не дает математического прироста точности, но кратно увеличивает вычислительную нагрузку, время предсказания и стоимость поддержки кода. Руководствуясь бизнес-логикой и показателем ROI, в финальный пайплайн внедрена одиночная модель **LightGBM**.

Также были получены индексы главных ошибок модели, например Лахта-центр, они были временно удалены из Train, чтобы модель не училась на плохих данных 

In [9]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import geopandas as gpd
import pydeck as pdk
import json
from sklearn.model_selection import KFold
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from lightgbm import LGBMRegressor

In [4]:
INPUT_CSV = "data/processed/CATBOOST_READY_CLEAN_FLOORS.csv"
OUTPUT_CSV = "submissions/FINAL_SUBMISSION_LGBM_3D.csv"
TARGET = "target_height"
RANDOM_STATE = 42
N_SPLITS = 5

# Те самые 8 зданий, которые ломают обучение (Лахта, вышки и т.д.)
BAD_IDS = [22196.0, 77650.0, 31536.0, 113305.0, 80618.0, 152373.0, 104813.0, 95834.0]

BASE_FEATURES = [
    "ml_category", "coord_x", "coord_y", "floor_count", 
    "actual_area_sqm", "spatial_cluster", "perimeter_m", "compactness", 
    "complex_total_area"
]

CATEGORICAL = ["ml_category", "spatial_cluster"]

# Найденные идеальные гиперпараметры
LGB_PARAMS = {
    'objective': 'regression_l1', 'metric': 'mae', 'random_state': RANDOM_STATE, 'verbose': -1, 'n_jobs': -1,
    'n_estimators': 2359,
    'learning_rate': 0.05552468672481143,
    'num_leaves': 78,
    'max_depth': 12,
    'min_child_samples': 27,
    'subsample': 0.9593884201060788,
    'colsample_bytree': 0.995856699728964,
    'reg_alpha': 7.880838703764818,
    'reg_lambda': 6.087470968947802,
}

# Функция для честного расчета высоты соседей
def get_dynamic_knn_height(train_coords, train_target, target_coords, is_train_set=False):
    # Если ищем соседей для самих себя (Train), берем k=6, чтобы потом откинуть себя же
    k = 6 if is_train_set else 5
    nn = NearestNeighbors(n_neighbors=k, n_jobs=-1)
    nn.fit(train_coords)
    
    distances, indices = nn.kneighbors(target_coords)
    
    if is_train_set:
        indices = indices[:, 1:] 
        
    # Вытаскиваем высоты по индексам и считаем среднее
    heights = train_target.iloc[indices.flatten()].values.reshape(indices.shape)
    return np.nanmean(heights, axis=1)


df_raw = pd.read_csv(INPUT_CSV)

# Приводим типы
for col in CATEGORICAL:
    if col in df_raw.columns:
        df_raw[col] = df_raw[col].fillna("Unknown").astype("category")

# Разделяем на Train и Test
train_mask = df_raw[TARGET].notna()
test_mask = df_raw[TARGET].isna()

df_train = df_raw[train_mask].copy()
df_test = df_raw[test_mask].copy()

# Изолируем наши 8 аномалий из обучающей выборки
df_anomalies = df_train[df_train['id'].isin(BAD_IDS)].copy()
df_train = df_train[~df_train['id'].isin(BAD_IDS)].reset_index(drop=True)

print(f"Строк в Train (без мусора): {len(df_train)}")
print(f"Строк в Test (надо заполнить): {len(df_test)}")
print(f"Изолировано аномалий: {len(df_anomalies)}")

# Кросс-валидация и обучение
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

oof_preds = np.zeros(len(df_train))
test_preds = np.zeros(len(df_test))

for fold, (train_idx, valid_idx) in enumerate(kf.split(df_train), start=1):
    
    # 1. Сплит данных
    train_fold = df_train.iloc[train_idx]
    valid_fold = df_train.iloc[valid_idx]
    
    y_train = train_fold[TARGET]
    y_valid = valid_fold[TARGET]
    
    X_train = train_fold[BASE_FEATURES].copy()
    X_valid = valid_fold[BASE_FEATURES].copy()
    X_test = df_test[BASE_FEATURES].copy()
    
    # 2. Честный расчет knn_mean_height_5
    coords_train = train_fold[['coord_x', 'coord_y']]
    coords_valid = valid_fold[['coord_x', 'coord_y']]
    coords_test = df_test[['coord_x', 'coord_y']]
    
    X_train['knn_mean_height_5'] = get_dynamic_knn_height(coords_train, y_train, coords_train, is_train_set=True)
    X_valid['knn_mean_height_5'] = get_dynamic_knn_height(coords_train, y_train, coords_valid, is_train_set=False)
    X_test['knn_mean_height_5'] = get_dynamic_knn_height(coords_train, y_train, coords_test, is_train_set=False)
    
    # Обучение
    model = LGBMRegressor(**LGB_PARAMS)
    model.fit(
        X_train, y_train, 
        eval_set=[(X_valid, y_valid)], 
        categorical_feature=CATEGORICAL, 
        callbacks=[]
    )
    
    # Предсказание
    valid_pred = model.predict(X_valid)
    oof_preds[valid_idx] = valid_pred
    
    # Для теста усредняем предсказания 5 моделей
    test_preds += model.predict(X_test) / N_SPLITS
    
    fold_mae = mean_absolute_error(y_valid, valid_pred)
    print(f"Fold {fold} MAE: {fold_mae:.4f}")

# Итоговые метрики
mae = mean_absolute_error(df_train[TARGET], oof_preds)
rmse = np.sqrt(mean_squared_error(df_train[TARGET], oof_preds))
r2 = r2_score(df_train[TARGET], oof_preds)

print("Финальные метрики:")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R2   : {r2:.4f}")


# Собираем куски обратно
df_train['pred_height'] = oof_preds
df_test['pred_height'] = test_preds

# У аномалий нет предсказаний, но у них есть реальный таргет
df_anomalies['pred_height'] = np.nan 

# Склеиваем всё в один DataFrame
df_final = pd.concat([df_train, df_test, df_anomalies], ignore_index=True)

# Защита от физического бреда: здание не может быть ниже 2.5 метров
df_final['pred_height'] = df_final['pred_height'].clip(lower=2.5)

# Создаем финальную колонку: берем оригинальную высоту, а если ее нет - берем предсказание
df_final['target_height_filled'] = df_final[TARGET].fillna(df_final['pred_height'])

# Возвращаем исходный порядок сортировки по ID 
df_final = df_final.sort_values(by='id').reset_index(drop=True)

# Сохраняем
df_final.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')

Строк в Train (без мусора): 110751
Строк в Test (надо заполнить): 53800
Изолировано аномалий: 8
Fold 1 MAE: 0.5038
Fold 2 MAE: 0.5077
Fold 3 MAE: 0.5506
Fold 4 MAE: 0.5108
Fold 5 MAE: 0.4950
Финальные метрики:
MAE  : 0.5136
RMSE : 2.5785
R2   : 0.9659


In [8]:
all_features_final = BASE_FEATURES + ['knn_mean_height_5']
importances_df = pd.DataFrame(index=all_features_final)
importances_df['total_gain'] = 0.0


fold_importance = model.boosted_info_['average_gain'] if hasattr(model, 'boosted_info_') else model.feature_importances_

importances_df['total_gain'] += model.feature_importances_ / N_SPLITS

# Считаем проценты
importances_df['percentage'] = (importances_df['total_gain'] / importances_df['total_gain'].sum()) * 100
importances_df = importances_df.sort_values(by='percentage', ascending=False)

print("\nВажность признаков")
print(importances_df[['percentage']].round(2))


Важность признаков
                    percentage
floor_count              18.12
coord_y                  14.87
coord_x                  11.10
knn_mean_height_5        10.56
compactness              10.39
spatial_cluster           9.12
perimeter_m               8.71
actual_area_sqm           7.86
complex_total_area        5.15
ml_category               4.13


### Строим нащу 3-D модель Санкт-Петербурга

In [ ]:
df_preds = pd.read_csv('submissions/FINAL_SUBMISSION_LGBM_3D.csv')
df_geom = pd.read_csv('data/processed/FINAL_DATASET_ABSOLUTE.csv', low_memory=False)

# стык по id
df_preds['id'] = pd.to_numeric(df_preds['id'], errors='coerce')
df_geom['id'] = pd.to_numeric(df_geom['id'], errors='coerce')

print(f"Строк с предсказаниями: {len(df_preds)}")
df_merged = pd.merge(df_preds, df_geom[['id', 'geometry']], on='id', how='inner')
print(f"Строк после склейки с геометрией: {len(df_merged)}")

if len(df_merged) == 0:
    print("Ошибка слияния: ID не совпали вообще.")
    exit()

# Геометрия
gdf = gpd.GeoDataFrame(df_merged, geometry=gpd.GeoSeries.from_wkt(df_merged['geometry']))
gdf.set_crs(epsg=4326, inplace=True) 

# Подготовка слоя

if 'target_height' in gdf.columns:
    original_col = 'target_height'
elif 'target_height_x' in gdf.columns: 
    original_col = 'target_height_x'
elif 'target_height_y' in gdf.columns:
    original_col = 'target_height_y'
else:
    original_col = None

if original_col:
    gdf['height_type'] = np.where(gdf[original_col].isna(), 'Предсказанная высота', 'Фактическая высота')
else:
    gdf['height_type'] = 'Высота'

gdf = gdf.fillna('Unknown')

# Раскраска по итоговой высоте
def get_color(height):
    try:
        h = float(height)
        if h < 15: return [50, 100, 255]       # Синий
        elif h < 35: return [50, 255, 150]     # Зеленый
        elif h < 75: return [255, 200, 50]     # Желтый
        else: return [255, 50, 50]             # Красный
    except:
        return [100, 100, 100]                 # Серый

gdf['fill_color'] = gdf['target_height_filled'].apply(get_color)

# Конвертируем в формат GeoJSON
geojson_data = json.loads(gdf.to_json())

# Создаем 3D-слой с полигонами 
layer = pdk.Layer(
    "GeoJsonLayer",
    geojson_data,
    opacity=0.9,
    stroked=True,            
    filled=True,             
    extruded=True,           
    wireframe=True,          
    get_elevation="properties.target_height_filled", 
    elevation_scale=1,       
    get_fill_color="properties.fill_color",          
    pickable=True            
)

# камера и рендер
view_state = pdk.ViewState(
    longitude=30.3158,
    latitude=59.9390,
    zoom=12,
    pitch=60,  
    bearing=-20 
)

tooltip = {
    "html": "<b>ID здания:</b> {id} <br/>"
            "<b>Категория:</b> {ml_category} <br/>"
            "<b>Площадь:</b> {actual_area_sqm} м² <br/>"
            "<b>{height_type}:</b> <span style='color: white; background: #e74c3c; padding: 2px 4px; border-radius: 3px;'>{target_height_filled} м</span>",
    "style": {"backgroundColor": "#2c3e50", "color": "white"}
}

r = pdk.Deck(
    layers=[layer],
    initial_view_state=view_state,
    tooltip=tooltip,
    map_style=pdk.map_styles.CARTO_DARK 
)

r.to_html('submissions/3D_LightGBM_FINAL.html')

Строк с предсказаниями: 164559
Строк после склейки с геометрией: 164559
